# Setup

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

In [ ]:
# Read all symbols
files = glob.glob("stck_data/*.csv")

list_df = []
for file in files:
    df_temp = pd.read_csv(file)
    symbol = os.path.basename(file).split('.')[0]
    df_temp['Symbol'] = symbol
    list_df.append(df_temp)

df = pd.concat(list_df, ignore_index=True)

In [ ]:
# FILE_PATH = "VNINDEX.csv"

# df = pd.read_csv(FILE_PATH)
# df['Symbol'] = 'VNINDEX' 

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
numeric_cols = ['Price', 'Open', 'High', 'Low', 'Vol.', 'Change %']

df[numeric_cols] = df[numeric_cols].ffill().bfill()

for col in numeric_cols:
    df[col] = df[col].str.replace(',', '', regex=True)
    df[col] = df[col].str.replace(' ', '', regex=False)


df['Vol.'] = (
    df['Vol.']
    .replace({'K': '*1e3', 'M': '*1e6', 'B': '*1e9'}, regex=True)
    .map(pd.eval)
    .astype(float)
)

In [ ]:
df.rename(
    columns={
        'Price': 'Close',
        'Change %': 'Change',
        'Vol.': 'Volume',
        'Open': 'Open',
        'High': 'High',
        'Low': 'Low'
    },
    inplace=True
)

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df['Change'] = df['Change'].str.replace('%', '', regex=True).astype(float)

for col in ['Close', 'Open', 'High', 'Low', 'Change']:
    df[col] = df[col].astype(float)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

Fill data with linear interpolation

In [ ]:
start_date = '2018-01-01'
end_date   = '2024-12-31'
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

filled_data = []

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values('Date').set_index('Date')
    group = group.loc[start_date:end_date]
    
    group = group.reindex(full_range)
    group.index.name = 'Date'
    
    group['Symbol'] = symbol

    numeric_cols = ['Close', 'Open', 'High', 'Low', 'Volume', 'Change']
    group[numeric_cols] = group[numeric_cols].apply(pd.to_numeric, errors='coerce')
    
    group[numeric_cols] = group[numeric_cols].interpolate(method='linear').ffill().bfill()
    
    filled_data.append(group)

df = pd.concat(filled_data).reset_index()
df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)

In [ ]:
len(df)    

In [ ]:
df.columns

In [ ]:
# Define the helper functions used below
def stochastic_oscillator(high, low, close, period=14):
    lowest_low = low.rolling(window=period, min_periods=1).min()
    highest_high = high.rolling(window=period, min_periods=1).max()
    return (close - lowest_low) / (highest_high - lowest_low) * 100

def true_range(high, low, close):
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs()
    ], axis=1).max(axis=1)
    return tr

def RSI_ex(series, period=14):
    delta = series.diff(1)
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(span=period, adjust=False).mean()
    avg_loss = loss.ewm(span=period, adjust=False).mean()
    RS = avg_gain / avg_loss
    return 100 - (100 / (1 + RS))

def stochastic_RSI_ex(rsi_series, period=14):
    min_rsi = rsi_series.rolling(window=period, min_periods=1).min()
    max_rsi = rsi_series.rolling(window=period, min_periods=1).max()
    return (rsi_series - min_rsi) / (max_rsi - min_rsi)

def weighted_moving_average(prices, window):
    return prices.rolling(window, min_periods=1).apply(
        lambda x: np.dot(x, np.arange(1, len(x)+1)) / np.sum(np.arange(1, len(x)+1)),
        raw=True
    )
    
def calculate_adx(high, low, close, period=14):
    prev_high = high.shift(1)
    prev_low = low.shift(1)
    prev_close = close.shift(1)
    up_move = high - prev_high
    down_move = prev_low - low
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    tr = true_range(high, low, close)
    atr = tr.rolling(window=period, min_periods=period).sum()
    plus_dm_sum = pd.Series(plus_dm).rolling(window=period, min_periods=period).sum()
    minus_dm_sum = pd.Series(minus_dm).rolling(window=period, min_periods=period).sum()
    plus_di = 100 * (plus_dm_sum / atr)
    minus_di = 100 * (minus_dm_sum / atr)
    dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
    adx = dx.rolling(window=period, min_periods=period).mean()
    # Here, DMI is defined as the difference between +DI and -DI.
    dmi = plus_di - minus_di
    return adx, plus_di, minus_di, dmi

def MFI(high, low, close, volume, period=14):
    typical_price = (high + low + close) / 3.0
    money_flow = typical_price * volume
    tp_diff = typical_price.diff(1)
    pos_mf = money_flow.where(tp_diff > 0, 0)
    neg_mf = money_flow.where(tp_diff < 0, 0)
    pos_mf_sum = pos_mf.rolling(window=period, min_periods=1).sum()
    neg_mf_sum = neg_mf.rolling(window=period, min_periods=1).sum().abs()
    return 100 * pos_mf_sum / (pos_mf_sum + neg_mf_sum)

def CCI(high, low, close, period):
    tp = (high + low + close) / 3.0
    sma_tp = tp.rolling(window=period, min_periods=1).mean()
    mad = tp.rolling(window=period, min_periods=1).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    return (tp - sma_tp) / (0.015 * mad)

def TRIX(prices, period=15):
    ema1 = prices.ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    return ema3.pct_change() * 100

def WILLR(high, low, close, period=14):
    highest_high = high.rolling(window=period, min_periods=1).max()
    lowest_low = low.rolling(window=period, min_periods=1).min()
    return -100 * (highest_high - close) / (highest_high - lowest_low)

In [ ]:
full_range = pd.date_range(start='2018-01-01', end='2024-12-31', freq='D')

features_list = []

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values('Date').set_index('Date')
    group = group.reindex(full_range)
    group.index.name = 'Date'
    group['Symbol'] = symbol
    
    # Returns based on Close
    group['return_day'] = group['Close'].pct_change()
    group['return_week'] = group['Close'].pct_change(periods=5)
    group['return_month'] = group['Close'].pct_change(periods=22)
    
    # Volatility: rolling std of daily returns
    group['volatility_day'] = group['return_day'].rolling(window=5, min_periods=1).std()
    group['volatility_week'] = group['return_day'].rolling(window=21, min_periods=1).std()
    group['volatility_month'] = group['return_day'].rolling(window=63, min_periods=1).std()
    
    # Liquidity: rolling mean of Volume
    group['liquidity_day'] = group['Volume'].rolling(window=5, min_periods=1).mean()
    group['liquidity_week'] = group['Volume'].rolling(window=21, min_periods=1).mean()
    group['liquidity_month'] = group['Volume'].rolling(window=63, min_periods=1).mean()
    
    # Pressure features
    group['high_minus_close'] = (group['High'] - group['Close']) / group['Open']
    group['low_minus_open'] = (group['Low'] - group['Open']) / group['Open']
    
    # Cumulative return
    group['cumulative_return'] = group['Close'] / group['Close'].iloc[0] - 1
    
    # Stochastic oscillator over 14-day window
    group['Stochastic_Osc'] = stochastic_oscillator(group['High'], group['Low'], group['Close'], period=14)
    
    # ATR (Average True Range) over 14-day window
    group['True_Range'] = true_range(group['High'], group['Low'], group['Close'])
    group['ATR'] = group['True_Range'].rolling(window=14, min_periods=1).mean()
    group.drop('True_Range', axis=1, inplace=True)
    
    # 'ADX14' and 'ADX20'
    # adx14, plus_di_14, minus_di_14, dmi14 = calculate_adx(group['High'], group['Low'], group['Close'], period=14)
    # adx20, plus_di_20, minus_di_20, dmi20 = calculate_adx(group['High'], group['Low'], group['Close'], period=20)
    # group['ADX14'] = adx14
    # group['ADX20'] = adx20
    
    # Simple Moving Averages (SMA) for windows 7, 14, 21, 50, 100
    sma_windows = [3, 7, 14, 21, 50, 100]
    for window in sma_windows:
        group[f'SMA_{window}'] = group['Close'].rolling(window, min_periods=1).mean()
    
    # Weighted Moving Averages (WMA) for windows 7, 14, 21, 50, 100
    wma_windows = [3, 7, 14, 21, 50, 100]
    for window in wma_windows:
        group[f'WMA_{window}'] = weighted_moving_average(group['Close'], window)
    
    # 11. EMA, MACD, and Signal computed using EMAs on Close
    group['EMA6'] = group['Close'].ewm(span=6, adjust=False).mean()
    group['EMA12'] = group['Close'].ewm(span=12, adjust=False).mean()
    group['EMA26'] = group['Close'].ewm(span=26, adjust=False).mean()
    group['outMACD'] = group['EMA12'] - group['EMA26']
    group['outMACDSignal'] = group['outMACD'].ewm(span=9, adjust=False).mean()
    group['outMACDHist'] = group['outMACD'] - group['outMACDSignal']    
    
    # 12. RSI computed on Close (exponential version, 14-day)
    group['RSI6'] = RSI_ex(group['Close'], period=6)
    group['RSI12'] = RSI_ex(group['Close'], period=12)
    group['RSI14'] = RSI_ex(group['Close'], period=14)
    
    # 13. Stochastic RSI on the RSI computed above
    group['StochRSI_6'] = stochastic_RSI_ex(group['RSI6'], 6)
    group['StochRSI_12'] = stochastic_RSI_ex(group['RSI12'], 12)
    group['StochRSI_14'] = stochastic_RSI_ex(group['RSI14'], 14)
    
    # 14. Bollinger Bands (21-day SMA ± 2 std deviations) on Close
    group['BBANDSMIDDLE'] = group['Close'].rolling(window=21, min_periods=1).mean()
    rolling_std = group['Close'].rolling(window=21, min_periods=1).std()
    group['BBANDSUPPER'] = group['BBANDSMIDDLE'] + 2 * rolling_std
    group['BBANDSLOWER'] = group['BBANDSMIDDLE'] - 2 * rolling_std
    
    # on balance volume
    obv = group['Volume'].copy() * 0  # initialize to 0
    price_diff = group['Close'].diff()
    obv = np.where(price_diff > 0, group['Volume'], np.where(price_diff < 0, -group['Volume'], 0))
    group['OBV'] = pd.Series(obv, index=group.index).cumsum()
    
    # money flow index
    group['MFI14'] = MFI(group['High'], group['Low'], group['Close'], group['Volume'], period=14)
    
    # momentum
    group['MOM1'] = group['Close'] - group['Close'].shift(1)
    group['MOM3'] = group['Close'] - group['Close'].shift(3)
    group['MOM7'] = group['Close'] - group['Close'].shift(7)
    
    # 'CCI12' and 'CCI20'
    group['CCI12'] = CCI(group['High'], group['Low'], group['Close'], period=12)
    group['CCI20'] = CCI(group['High'], group['Low'], group['Close'], period=20)
    
    # 'ROCR3' and 'ROCR12' – Rate Of Change Ratio
    group['ROCR3'] = group['Close'] / group['Close'].shift(3)
    group['ROCR12'] = group['Close'] / group['Close'].shift(12)
    
    # 'WILLR' – Williams %R (using period 14)
    group['WILLR'] = WILLR(group['High'], group['Low'], group['Close'], period=14)
                            
    # 'TRIX' – Triple Exponential Moving Average Rate Of Change (using period 15)
    group['TRIX'] = TRIX(group['Close'], period=15)
    
    features_list.append(group)

In [ ]:
df = pd.concat(features_list).reset_index().rename(columns={'index': 'Date'})
df = df.sort_values(['Symbol', 'Date']).reset_index(drop=True)

# Outliers

In [ ]:
# First, calculate z-scores if you haven't already
def add_z_score(group):
    mean = group['Price'].mean()
    std = group['Price'].std()
    group['z_score'] = (group['Price'] - mean) / std
    return group

df.reset_index(inplace=True) 

df = df.groupby('Symbol', group_keys=False).apply(add_z_score)

In [ ]:
symbols = df['Symbol'].unique()

for symbol in symbols:
    df_symbol = df[df['Symbol'] == symbol]

    plt.figure(figsize=(14, 6))
    
    # Plot Price clearly
    plt.plot(df_symbol['Date'], df_symbol['Price'], color='blue', label='Price')

    # Identify and plot Outliers (z_score > ±3)
    outliers = df_symbol[np.abs(df_symbol['z_score']) > 3]
    plt.scatter(outliers['Date'], outliers['Price'], color='red', label='Outliers')

    plt.title(f'Price and Outliers for Symbol: {symbol}')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# # remove outliers
# df = df[df['z_score'].abs() < 3].copy()
# df.drop(columns=['z_score'], inplace=True)
# df.set_index(['Symbol', 'Date'], inplace=True)

# Further Processing

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
# Fill missing values with forward and backward fill
df = df.ffill().bfill()

In [ ]:
df.columns

In [ ]:
len(df)

In [ ]:
df.to_csv('processed_stock_data.csv', index=False)

# Micro/Marco Indicators

In [ ]:
def clean_macro_csv(file_path):
    df = pd.read_csv(file_path, skiprows=4)
    df.dropna(axis=1, how='all', inplace=True)

    df_long = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
        var_name="Year",
        value_name="Value"
    )
    
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce").astype('Int64')
    df_long["Value"] = pd.to_numeric(df_long["Value"], errors="coerce")
    
    df_long = df_long.reset_index(drop=True)
    
    return df_long

gdp_file = "stck_data/Chỉ số vĩ mô/GDP/GDP.csv"
gdp_growth_file = 'stck_data/Chỉ số vĩ mô/GDP_growth/GDP_growth.csv'
gdp_per_capita_file = 'stck_data/Chỉ số vĩ mô/GDP_per_capita/GDP_per_capita.csv'
inflation_file = 'stck_data/Chỉ số vĩ mô/Inflation/Inflation.csv'
unemployment_file = 'stck_data/Chỉ số vĩ mô/Unemployment/Unemployment.csv'

gdp_df = clean_macro_csv(gdp_file)
gdp_growth_df = clean_macro_csv(gdp_growth_file)
gdp_per_capita_df = clean_macro_csv(gdp_per_capita_file)
inflation_df = clean_macro_csv(inflation_file)
unemployment_df = clean_macro_csv(unemployment_file)

target_country = "Viet Nam"

vn_df = gdp_df[gdp_df["Country Name"].str.strip().str.lower() == target_country.lower()]    
vn_gdp_growth = gdp_growth_df[gdp_growth_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_gdp_per_capita = gdp_per_capita_df[gdp_per_capita_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_inflation = inflation_df[inflation_df["Country Name"].str.strip().str.lower() == target_country.lower()]
vn_unemployment = unemployment_df[unemployment_df["Country Name"].str.strip().str.lower() == target_country.lower()]

In [ ]:
vn_gdp_growth = vn_gdp_growth.dropna()
vn_gdp_per_capita  = vn_gdp_per_capita .dropna()
vn_df = vn_df.dropna()
vn_inflation = vn_inflation.dropna()
vn_unemployment = vn_unemployment.dropna()

In [ ]:
vn_df = vn_df.rename(columns={"Value": "GDP"})
vn_gdp_growth = vn_gdp_growth.rename(columns={"Value": "GDP_Growth"})
vn_gdp_per_capita = vn_gdp_per_capita.rename(columns={"Value": "GDP_Per_Capita"})
vn_inflation = vn_inflation.rename(columns={"Value": "Inflation"})
vn_unemployment = vn_unemployment.rename(columns={"Value": "Unemployment"})

In [ ]:
vn_merged = vn_df[["Year", "GDP"]].merge( # có từ 1985
    vn_gdp_growth[["Year", "GDP_Growth"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_gdp_per_capita[["Year", "GDP_Per_Capita"]], # có từ 1985
    on="Year",
    how="inner"
).merge(
    vn_inflation[["Year", "Inflation"]], # có từ 1996
    on="Year",
    how="inner"
).merge(
    vn_unemployment[["Year", "Unemployment"]], # có từ 1991
    on="Year",
    how="inner"
)

vn_merged = vn_merged.sort_values("Year").reset_index(drop=True)

In [ ]:
vn_merged

In [ ]:
vn_merged.to_csv("vietnam_macro.csv", index=False)